In [15]:
# ============================================================
# 03_vectorization.ipynb — BoW, TF-IDF, Train/Val/Test Split
# ============================================================

In [16]:
import pandas as pd
import pickle
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split

In [17]:
# ── 1. LOAD & CLEAN ─────────────────────────────────────────
df = pd.read_csv("../data/processed/clean_reviews.csv")

# clean_text  → used for BoW / TF-IDF
df['clean_text'] = df['clean_text'].fillna('').astype(str)
df = df[df['clean_text'].str.strip() != '']

# bert_text   → used for DistilBERT (raw text, only HTML stripped)
df['bert_text'] = df['bert_text'].fillna('').astype(str)

print("✅ Clean rows:", len(df))

✅ Clean rows: 709313


In [18]:
# ── 2. SPLIT ────────────────────────────────────────────────
X_bow  = df['clean_text']   # for BoW / TF-IDF
X_bert = df['bert_text']    # for DistilBERT
y = df['label']

# Shared index split — same rows across both feature sets
idx_train, idx_temp, y_train, y_temp = train_test_split(
    df.index, y, test_size=0.3, random_state=42, stratify=y
)
idx_val, idx_test, y_val, y_test = train_test_split(
    idx_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

X_train_bow  = X_bow.loc[idx_train]
X_val_bow    = X_bow.loc[idx_val]
X_test_bow   = X_bow.loc[idx_test]

X_train_bert = X_bert.loc[idx_train]
X_val_bert   = X_bert.loc[idx_val]
X_test_bert  = X_bert.loc[idx_test]

print(f"✅ Train: {len(idx_train)} | Val: {len(idx_val)} | Test: {len(idx_test)}")

# Save CSV splits — clean_text for classical ML, bert_text for DistilBERT
print("⏳ Saving CSV files...")

X_train_bow.to_frame(name='clean_text').assign(
    bert_text=X_train_bert.values, label=y_train.values
).to_csv("../data/processed/train.csv", index=False)

X_val_bow.to_frame(name='clean_text').assign(
    bert_text=X_val_bert.values, label=y_val.values
).to_csv("../data/processed/val.csv", index=False)

X_test_bow.to_frame(name='clean_text').assign(
    bert_text=X_test_bert.values, label=y_test.values
).to_csv("../data/processed/test.csv", index=False)

print("✅ CSV files (Train/Val/Test) saved successfully!")

✅ Train: 496519 | Val: 106397 | Test: 106397
⏳ Saving CSV files...
✅ CSV files (Train/Val/Test) saved successfully!


In [19]:
# ── 3. BAG OF WORDS ─────────────────────────────────────────
print("⏳ Fitting BoW... (may take ~1 min)")
bow_vectorizer = CountVectorizer(max_features=5000)
X_train_bow_mat = bow_vectorizer.fit_transform(X_train_bow)
X_val_bow_mat   = bow_vectorizer.transform(X_val_bow)
X_test_bow_mat  = bow_vectorizer.transform(X_test_bow)
print("✅ BoW done:", X_train_bow_mat.shape)

⏳ Fitting BoW... (may take ~1 min)
✅ BoW done: (496519, 5000)


In [20]:
# ── 4. TF-IDF ───────────────────────────────────────────────
print("⏳ Fitting TF-IDF... (may take ~1 min)")
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_bow)
X_val_tfidf   = tfidf_vectorizer.transform(X_val_bow)
X_test_tfidf  = tfidf_vectorizer.transform(X_test_bow)
print("✅ TF-IDF done:", X_train_tfidf.shape)

⏳ Fitting TF-IDF... (may take ~1 min)
✅ TF-IDF done: (496519, 5000)


In [21]:
# ── 5. EXPORT ───────────────────────────────────────────────
print("⏳ Saving files...")

with open("../data/vectorized/bow_vectorizer.pkl", "wb") as f:
    pickle.dump(bow_vectorizer, f)
with open("../data/vectorized/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_vectorizer, f)

# BoW matrices
with open("../data/vectorized/X_train_bow.pkl", "wb") as f:
    pickle.dump((X_train_bow_mat, y_train), f)
with open("../data/vectorized/X_val_bow.pkl", "wb") as f:
    pickle.dump((X_val_bow_mat, y_val), f)
with open("../data/vectorized/X_test_bow.pkl", "wb") as f:
    pickle.dump((X_test_bow_mat, y_test), f)

# TF-IDF matrices
with open("../data/vectorized/X_train_tfidf.pkl", "wb") as f:
    pickle.dump((X_train_tfidf, y_train), f)
with open("../data/vectorized/X_val_tfidf.pkl", "wb") as f:
    pickle.dump((X_val_tfidf, y_val), f)
with open("../data/vectorized/X_test_tfidf.pkl", "wb") as f:
    pickle.dump((X_test_tfidf, y_test), f)

# DistilBERT raw text splits (saved in CSV already, also pickle for convenience)
with open("../data/vectorized/bert_splits.pkl", "wb") as f:
    pickle.dump({
        'X_train': X_train_bert.values, 'y_train': y_train.values,
        'X_val':   X_val_bert.values,   'y_val':   y_val.values,
        'X_test':  X_test_bert.values,  'y_test':  y_test.values,
    }, f)

print("🎉 All files exported successfully!")

⏳ Saving files...
🎉 All files exported successfully!
